# OMNI MOVIE STUDIO — Free GPU (Colab T4)\nFree animation + lip-sync + Hindi TTS for the omni-media-agent Movie Mode. No paid APIs. Files save to Google Drive.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null\nimport os, glob\nfrom google.colab import drive\ndrive.mount('/content/drive')\nOUT = '/content/drive/MyDrive/omni-movie'\nos.makedirs(OUT, exist_ok=True)\nprint('ready:', OUT)

In [ ]:
# ComfyUI + Wan 2.2 1.3B T2V (runs on T4; for I2V swap in the 5B fp8 model + CLIP Vision)\n!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI\n%pip -q install -r /content/ComfyUI/requirements.txt torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121\n!wget -q -P /content/ComfyUI/models/checkpoints https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repoduced_and_Fixed/resolve/main/split_files/diffusion_models/wan2.2_t2v_high_noise_1.3B_fp16.safetensors 2>/dev/null || echo 'download models per ComfyUI docs'\n%cd /content/ComfyUI\nimport subprocess, threading\nthreading.Thread(target=lambda: subprocess.run(['python','main.py','--port','8188','--dont-print-server']), daemon=True).start()\nimport time; time.sleep(30)\nimport urllib.request; print(urllib.request.urlopen('http://127.0.0.1:8188/system_stats').read()[:80])

In [ ]:
# Wav2Lip — free lip-sync (runs even on T4/CPU)\n!git clone --depth 1 https://github.com/Rudrabha/Wav2Lip /content/Wav2Lip\n%cd /content/Wav2Lip\n%pip -q install librosa==0.10.1 numba==0.58.1\n!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/wav2lip_gan.pth -O checkpoints/wav2lip_gan.pth\n!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/s3fd.pth -O face_detection/detection/sfd/s3fd.pth\nprint('wav2lip ready')

In [ ]:
# XTTS v2 — expressive Hindi TTS\n%pip -q install TTS==0.22.0\n# quick test:\nfrom TTS.api import TTS\ntts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')\ntts.tts_to_file(text='मैं जन्म से मृत्यु तक ब्रह्मचारी रहूँगा!', language='hi', speaker_wav='reference.wav', file_path=f'{OUT}/devavrata.wav')  # put any Hindi reference clip\nprint('xtts ready')

In [ ]:
# ANIMATE: point COMFYUI_URL / WAN_I2V_WORKFLOW at this instance from omni-media-agent,\n# or batch-render shot stills+prompts here, then zip results back to Drive:\n!zip -r {OUT}/omni_movie_output.zip /content/ComfyUI/output/*\nprint('done — check Drive/omni-movie')